In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def get_latency_and_arrival(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    latency_list = []
    executor_list = []
    arrival_list = []
    for item in data:
        if item["response"]["meta_data"]["finish_reason"] not in ["stop", "length"]:
            continue
        latency = item["timestamp_list"][-1] - item["timestamp_list"][0]
        arrival = item["timestamp_list"][0]
        latency_list.append(latency)
        executor_list.append(item["response"]["executor_node"])
        arrival_list.append(arrival)

    return np.array(latency_list), executor_list, np.array(arrival_list)

In [ ]:
result_folder = "../results/decentralized_simulation"
result_path = f"{result_folder}/result.json"

latency, executor_list, arrival_times = get_latency_and_arrival(result_path)

t0 = np.min(arrival_times)
arrival_times = (arrival_times - t0)

sorted_idx = np.argsort(arrival_times)
arrival_times_sorted = arrival_times[sorted_idx]
latency = latency[sorted_idx]
executor_list = [executor_list[i] for i in sorted_idx]

In [ ]:
colors = {
    "node1": "#A0A0A0",
    "node2": "#7FB0C0",
    "node3": "#B0C070",
    "node4": "#D95F02"
}

plt.figure(figsize=(8, 4))

for i, (lat, node) in enumerate(zip(latency, executor_list)):
    plt.bar(i, lat, color=colors[node], width=0.7, alpha=0.8)

window_size = 45
latency_avg = pd.Series(latency).rolling(window=window_size, center=True, min_periods=1).mean().to_numpy()
x_avg = np.arange(len(latency))

plt.plot(x_avg, latency_avg, color="black", linewidth=1)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel("Request Index", fontsize=14)
plt.ylabel("Latency (s)", fontsize=14)
plt.legend(handles=[plt.Line2D([0], [0], color=color, lw=2, label=node) for node, color in colors.items()], fontsize=10)
plt.tight_layout()
plt.show()